<a href="https://colab.research.google.com/github/kylejgillett/sounderpy/blob/main/colab_tutorials/field_campaign_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Field Campaign Sounding Viewer
### Powered by [SounderPy](https://sounderpy.app/)

Plot a sounding from your own launch in a few clicks: upload a SHARPpy-format text file (the `%TITLE%` / `%RAW%` format SHARPpy reads) and get a full SounderPy sounding.

**How to use it**
1. Run **Step 1** once per session.
2. Fill in **Step 2** (your campaign and launch site) and run it.
3. For each launch, run **Step 3** (upload) and **Step 4** (plot). *Runtime → Run all* runs everything.

Click ▶ at the left of a cell, or press **Shift + Enter**, to run it.

📖 [SounderPy docs](https://sounderpy.app/docs/) · [Reading SHARPpy files](https://sounderpy.app/docs/reference/customdatasources.html#ingesting-sharppy-files) · [Sounding options](https://sounderpy.app/docs/reference/plottingdata.html#soundings) · [Troubleshooting](https://sounderpy.app/docs/user_guide/troubleshooting.html) · [Questions & community](https://sounderpy.app/docs/user_guide/community.html)

In [ ]:
# @title **Step 1 · Install SounderPy** (once per session, about a minute)
# @markdown If Colab asks to restart the session after installing, click **Restart session** and run this cell again.

%pip install -q "sounderpy>=3.3.0"

import re
import sys
import warnings
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import sounderpy as spy
from sounderpy.utils import SounderPyNotice

# this notebook fills in the site meta-data and titles, so skip the reminder to add them
warnings.filterwarnings("ignore", category=SounderPyNotice)

IN_COLAB = "google.colab" in sys.modules


def _number(text, name):
    """An optional number from a settings field: None when blank."""
    text = str(text).strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        raise ValueError(f"{name} must be a number, not {text!r}. Fix it in Step 2 and run Step 2 again.") from None


def _storm_motion(text):
    """'right_moving', 'left_moving', 'mean_wind', or a custom 'direction/speed' such as '230/40' (deg/kt)."""
    text = str(text).strip()
    parts = re.split(r"[\s/,]+", text)
    if len(parts) == 2:
        return [_number(parts[0], "STORM_MOTION direction"), _number(parts[1], "STORM_MOTION speed")]
    return text or "right_moving"


def upload_file():
    """Ask for a file with Colab's upload button and return its path."""
    if not IN_COLAB:
        raise ValueError("Outside of Colab, set FILE_PATH in Step 3 to your sounding file.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded. Run Step 3 again and choose a file.")
    return list(uploaded)[-1]


def _valid_time(text):
    """An optional 'YYYY-MM-DD HH:MM' (UTC) from a settings field: None when blank."""
    text = str(text).strip()
    if not text:
        return None
    try:
        return datetime.strptime(text, "%Y-%m-%d %H:%M")
    except ValueError:
        raise ValueError(f"VALID_TIME must look like 2026-05-21 18:45, not {text!r}. "
                         "Fix it in Step 2 and run Step 2 again.") from None


def _time_label(site, valid):
    """'VALID: 05/21/2026 18:45Z' from Step 2's VALID_TIME, or 'VALID: 05/21/2026 18Z' from the file."""
    if valid:
        return f"VALID: {valid:%m/%d/%Y %H:%M}Z"
    if site.get("valid-time"):
        year, month, day, hour = (int(v) for v in site["valid-time"][:4])
        return f"VALID: {month:02d}/{day:02d}/{year} {hour:02d}Z"
    return "VALID TIME UNKNOWN"


def read_sounding(path):
    """Read the SHARPpy file, add the site meta-data from Step 2, and build the plot titles."""
    lat, lon = _number(SITE_LAT, "SITE_LAT"), _number(SITE_LON, "SITE_LON")
    if lat is None or lon is None:
        raise ValueError("SHARPpy files hold no launch lat/lon: set SITE_LAT and SITE_LON in Step 2, "
                         "run Step 2, then run Step 3 again.")
    elevation = _number(SITE_ELEVATION_M, "SITE_ELEVATION_M")
    valid = _valid_time(VALID_TIME)

    try:
        data = spy.from_sharppy(path)
    except (ValueError, UnicodeDecodeError):
        raise ValueError(f"{Path(path).name} isn't a SHARPpy file that SounderPy can read. For other formats, see "
                         "https://sounderpy.app/docs/reference/customdatasources.html#creating-custom-clean-data") from None

    # Step 2's settings fill in or replace what the file holds
    site = data["site_info"]
    site["site-id"] = SITE_ID.strip().upper() or site.get("site-id") or ""
    site["site-name"] = SITE_NAME.strip().upper()
    site["site-lctn"] = ""
    site["site-latlon"] = [lat, lon]
    # SHARPpy heights are above sea level, so the lowest level is the launch elevation
    site["site-elv"] = elevation if elevation is not None else round(float(data["z"][0].m))
    if valid:
        site["valid-time"] = [valid.year, valid.month, valid.day, valid.hour]
    campaign = CAMPAIGN_NAME.strip().upper() or "FIELD CAMPAIGN"
    site["source"] = campaign
    site["data-source"] = "SHARPPY FILE"

    where = " - ".join(part for part in (site["site-id"], site["site-name"]) if part)
    where += f" | {site['site-latlon'][0]:.2f}, {site['site-latlon'][1]:.2f}"
    data["titles"] = {"top_title": f"{campaign} | OBSERVED VERTICAL PROFILE",
                      "left_title": _time_label(site, valid),
                      "right_title": f"{where}    "}

    # a short summary to check the file before plotting
    p, z = data["p"].m, data["z"].m
    print(f"✅ SHARPpy file: {Path(path).name}")
    print(f"   {_time_label(site, valid).title()} · {len(p)} levels · {p[0]:.0f} to {p[-1]:.0f} hPa "
          f"({(z[-1] - z[0]) / 1000:.1f} km deep)")
    if not site.get("valid-time"):
        print("   ⚠️ no valid time in the file: set VALID_TIME in Step 2 for the title and the radar")
    return data


def _download(filename):
    if IN_COLAB:
        from google.colab import files
        files.download(filename)
    print(f"💾 Saved {filename}")


def _file_stem(data):
    site = data["site_info"]
    vt, valid = site.get("valid-time"), _valid_time(VALID_TIME)
    when = (f"{valid:%Y%m%d_%H%M}" if valid
            else f"{int(vt[0])}{int(vt[1]):02d}{int(vt[2]):02d}_{int(vt[3]):02d}Z" if vt else "")
    stem = "_".join(part for part in (site["site-id"] or site["site-name"], when) if part) or "sounding"
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", stem)


def _plot_options(data):
    return dict(dark_mode=DARK_MODE, storm_motion=_storm_motion(STORM_MOTION),
                radar=None if RADAR == "none" else RADAR, map_zoom=2 if SHOW_MAP else 0)


def plot_sounding(data):
    """Draw the sounding with Step 2's settings, then save & download it when SAVE_FIGURES is on."""
    fig, _ = spy.build_sounding(data, color_blind=COLOR_BLIND, mode=MODE,
                                special_parcels=None if ECAPE_PARCEL else "simple",
                                return_fig=True, **_plot_options(data))
    if SAVE_FIGURES:
        filename = f"{_file_stem(data)}_sounding.png"
        fig.savefig(filename, bbox_inches="tight")
        _download(filename)
    plt.show()


def plot_hodograph(data):
    """Draw the hodograph figure with Step 2's settings, then save & download it when SAVE_FIGURES is on."""
    fig, _ = spy.build_hodograph(data, sr_hodo=STORM_RELATIVE, return_fig=True, **_plot_options(data))
    if SAVE_FIGURES:
        filename = f"{_file_stem(data)}_hodograph.png"
        fig.savefig(filename, bbox_inches="tight")
        _download(filename)
    plt.show()


print(f"✅ SounderPy {version('sounderpy')} is ready. Now fill in Step 2.")

In [ ]:
# @title **Step 2 · Campaign & plot settings**
# @markdown Change a setting, then run this cell again.

# @markdown ---
# @markdown **Campaign & launch site**
CAMPAIGN_NAME = "MY FIELD CAMPAIGN"  # @param {type:"string"}
SITE_ID = ""  # @param {type:"string"}
SITE_NAME = ""  # @param {type:"string"}
# @markdown **Required**: the launch lat/lon in decimal degrees, e.g. `47.92` and `-97.09`.
SITE_LAT = ""  # @param {type:"string"}
SITE_LON = ""  # @param {type:"string"}
# @markdown Meters above sea level. Blank uses the file's lowest height.
SITE_ELEVATION_M = ""  # @param {type:"string"}

# @markdown The launch time in UTC, as `YYYY-MM-DD HH:MM`. Blank uses the file's (date and hour only).
VALID_TIME = ""  # @param {type:"string"}

# @markdown ---
# @markdown **Plot** · [all sounding options](https://sounderpy.app/docs/reference/plottingdata.html#soundings)
# @markdown `auto` draws a [winter sounding](https://sounderpy.app/docs/tutorials/winter_soundings.html) when the surface wet-bulb is below 35 °F.
MODE = "auto"  # @param ["auto", "severe", "winter"]
# @markdown [Storm motion](https://sounderpy.app/docs/reference/plotsettings.html#storm-motion-logic): pick one, or type your own as `direction/speed` (deg/kt), e.g. `230/40`.
STORM_MOTION = "right_moving"  # @param ["right_moving", "left_moving", "mean_wind"] {allow-input: true}
# @markdown [Radar](https://sounderpy.app/docs/reference/plotsettings.html#radar-logic) on the map inset: the U.S. mosaic is only available for the last month. `none` is fastest on a slow connection.
RADAR = "mosaic"  # @param ["mosaic", "single", "none"]
SHOW_MAP = True  # @param {type:"boolean"}
# @markdown Also draw the [most-unstable ECAPE parcel](https://sounderpy.app/docs/reference/plotsettings.html#parcel-logic) (slower).
ECAPE_PARCEL = False  # @param {type:"boolean"}
DARK_MODE = False  # @param {type:"boolean"}
COLOR_BLIND = True  # @param {type:"boolean"}
# @markdown Save each figure as a PNG and download it.
SAVE_FIGURES = True  # @param {type:"boolean"}

_storm_motion(STORM_MOTION)
for _name in ("SITE_LAT", "SITE_LON", "SITE_ELEVATION_M"):
    _number(globals()[_name], _name)
_valid_time(VALID_TIME)
print("✅ Settings saved. Now run Step 3.")

In [ ]:
# @title **Step 3 · Upload & read the SHARPpy file**
# @markdown Run this cell and choose your file with the **Choose files** button that appears below.
# @markdown Or give a path instead, such as a file in Google Drive (`/content/drive/MyDrive/...`, after mounting Drive from the 📁 panel).
FILE_PATH = ""  # @param {type:"string"}

clean_data = read_sounding(FILE_PATH.strip() or upload_file())

In [ ]:
# @title **Step 4 · Plot the sounding**
plot_sounding(clean_data)

---
## Extras

Run these after Step 3.

* **Hodograph figure**: a larger hodograph with the map. [Hodograph options](https://sounderpy.app/docs/reference/plottingdata.html#hodographs)
* **Save the data**: a CSV, a SHARPpy file, or a CM1 `input_sounding`. [Exporting data](https://sounderpy.app/docs/tutorials/exporting_data.html)
* **Your own Python**: `clean_data` is a standard SounderPy profile; see [the `clean_data` scheme](https://sounderpy.app/docs/reference/clean_data_scheme.html) and [parameters](https://sounderpy.app/docs/reference/helpertools.html) to go further.

In [ ]:
# @title **Hodograph figure**
STORM_RELATIVE = False  # @param {type:"boolean"}

plot_hodograph(clean_data)

In [ ]:
# @title **Save the data to a file**
EXPORT_TYPE = "sharppy"  # @param ["sharppy", "csv", "cm1"]

_suffix = {"sharppy": "_sharppy.txt", "csv": ".csv", "cm1": "_input_sounding"}[EXPORT_TYPE]
_filename = f"{_file_stem(clean_data)}{_suffix}"
spy.to_file(EXPORT_TYPE, clean_data, filename=_filename)
_download(_filename)

---
## Troubleshooting

* **"isn't a SHARPpy file"**: the file needs SHARPpy's `%TITLE%` and `%RAW%` sections ([format](https://sounderpy.app/docs/reference/customdatasources.html#ingesting-sharppy-files)). Other formats can be read into SounderPy by hand: see [Creating custom `clean_data`](https://sounderpy.app/docs/reference/customdatasources.html#creating-custom-clean-data).
* **Slow plot or a weak connection**: set `RADAR` to `none` (or turn off `SHOW_MAP`) and leave `ECAPE_PARCEL` off. The map and radar are the only parts that download data.
* **Shallow profile** (a launch still in progress): the plot draws, but the Bunkers and mean-wind storm motions need winds through 6 km, so the storm-relative values stay blank (with warnings) until then. Use a custom `direction/speed` storm motion to fill them in.
* More: [SounderPy troubleshooting](https://sounderpy.app/docs/user_guide/troubleshooting.html)